# Clinical Validation of 3D Mammography AI — Duke BCS-DBT (FROC)

Standalone **lesion-localisation** validation on digital breast tomosynthesis (DBT) — 3D mammography — using the Duke *Breast-Cancer-Screening-DBT* dataset.

The headline 3D metric is **FROC** (free-response ROC): it scores every region *mark* a detector emits — a mark counts only when it lands on a real lesion under the official geometric criterion — and reports sensitivity at 1/2/3/4 false marks per volume. Their mean is the official Duke BCS-DBT / DBTex challenge ranking metric.

This notebook runs entirely on **small metadata files** (a few MB — no 1.5 TB DICOM download): the ground-truth lesion boxes, and the **real DBTex-challenge team submissions**. So it validates *actual* competition detectors, end-to-end, against the official metric.

*(Exam-level volume classification — running a model on the DICOM pixels — is a heavier separate effort; see the closing notes.)*

## 1 · Environment setup
Runs on **Colab** or **Kaggle**. Turn *Internet* **On** (to clone the project code and download the metadata). No GPU needed — FROC is pure metric computation on small CSVs.

In [ ]:
# Make the `mammoval` package importable. Auto-clones the project repo if it
# is not already present (Internet must be ON for the clone on Kaggle).
import os, sys, glob, subprocess

REPO_URL = 'https://github.com/Joana-Mansa/breast-ai-clinical-validation.git'

def _find_mammoval():
    cands = ['.', 'breast-ai-clinical-validation',
             '/content/breast-ai-clinical-validation']
    cands += glob.glob('/kaggle/input/*') + glob.glob('/kaggle/input/*/*')
    cands += glob.glob('/kaggle/working/*')
    for c in cands:
        if c and os.path.isdir(os.path.join(c, 'mammoval')):
            return os.path.abspath(c)
    return None

loc = _find_mammoval()
if loc is None:
    print('cloning', REPO_URL, '...')
    subprocess.run(['git', 'clone', '-q', REPO_URL], check=False)
    loc = _find_mammoval()
if loc is None:
    raise RuntimeError('mammoval not found and clone failed - turn Internet ON, '
                       'or add the project folder as a Kaggle dataset (+ Add Input).')
sys.path.insert(0, loc)
import mammoval
print('mammoval', mammoval.__version__, 'ready (from ' + loc + ')')

## 2 · Download the Duke metadata and DBTex predictions
All public, no login, a few MB total: the validation-split ground-truth CSVs, and the DBTex challenge team predictions (`team_predictions_bothphases.zip`, ~7 MB).

In [ ]:
import os, urllib.request
os.makedirs('data/duke-dbt', exist_ok=True)
BASE = 'https://www.cancerimagingarchive.net/wp-content/uploads/'
FILES = [
    'BCS-DBT-file-paths-validation-v2.csv',
    'BCS-DBT-labels-validation-PHASE-2-Jan-2024.csv',
    'BCS-DBT-boxes-validation-v2-PHASE-2-Jan-2024.csv',
    'team_predictions_bothphases.zip',
]
for name in FILES:
    dst = os.path.join('data/duke-dbt', name)
    if not os.path.exists(dst):
        urllib.request.urlretrieve(BASE + name, dst)
    print('%-46s %8.1f KB' % (name, os.path.getsize(dst) / 1024))

## 3 · Load the dataset
`DukeDBTDataset` reads the metadata CSVs into the standard case table. `require_local=False` because this run needs no DICOM pixels — only the per-view labels and the ground-truth lesion boxes.

In [ ]:
from mammoval.data import DukeDBTDataset
import json

ds = DukeDBTDataset(root='data/duke-dbt', split='validation',
                    require_local=False)
print(json.dumps(ds.summary(), indent=2, default=str))
n_boxes = sum(len(ds.lesion_boxes(c)) for c in ds.cases['case_id'])
print('ground-truth lesion boxes:', n_boxes)

## 4 · Detector predictions — DBTex challenge teams
The Duke group published the DBTex1/DBTex2 challenge submissions. `read_dbtex_predictions` loads one team's detections for this split and converts the boxes to the format the FROC engine consumes. The phase-2 validation set has three teams: `nyu_bteam`, `vicorob`, `zedus`.

In [ ]:
ZIP = 'data/duke-dbt/team_predictions_bothphases.zip'
TEAM = 'nyu_bteam'        # or 'vicorob', 'zedus'

preds = ds.read_dbtex_predictions(ZIP, team=TEAM)
print(TEAM, '->', len(preds), 'detections over',
      preds['case_id'].nunique(), 'volumes')
preds.head()

## 5 · FROC localisation validation
`run_localization_validation` matches each detection to a ground-truth lesion under the **official Duke criterion** (`duke_dbt_hit`: centre within `max(√(W²+H²)/2, 100)` px and within `VolumeSlices/4` slices), builds the FROC curve, and reports sensitivity at 1/2/3/4 FP per volume with a case-level bootstrap CI.

In [ ]:
from mammoval.metrics import duke_dbt_hit
from mammoval.pipeline import run_localization_validation

loc = run_localization_validation(
    ds.assemble_froc_cases(preds), duke_dbt_hit,
    fp_points=(1, 2, 3, 4),
    dataset_name='Duke BCS-DBT validation split')
print('sensitivity @ FP/volume:',
      {k: round(v, 3) for k, v in loc['sensitivity_at_fp'].items()})
print('mean sensitivity = %.3f  (95%% CI %.3f-%.3f)  [official Duke metric]'
      % (loc['mean_sensitivity'], *loc['mean_sensitivity_ci']))

### Compare all three challenge teams
The same engine ranks every team — exactly what a validation function does when comparing candidate detectors.

In [ ]:
import pandas as pd

rows = []
for team in ['nyu_bteam', 'vicorob', 'zedus']:
    p = ds.read_dbtex_predictions(ZIP, team=team)
    r = run_localization_validation(ds.assemble_froc_cases(p),
                                    duke_dbt_hit, fp_points=(1, 2, 3, 4),
                                    n_boot=500)
    rows.append({'team': team,
                 'mean_sensitivity': round(r['mean_sensitivity'], 3),
                 'CI_low': round(r['mean_sensitivity_ci'][0], 3),
                 'CI_high': round(r['mean_sensitivity_ci'][1], 3),
                 **{k: round(v, 3) for k, v in r['sensitivity_at_fp'].items()}})
pd.DataFrame(rows).sort_values('mean_sensitivity', ascending=False)

## 6 · Validation report

In [ ]:
from mammoval.report import build_report
from IPython.display import HTML

os.makedirs('outputs', exist_ok=True)
path = build_report(
    localization_results=loc,
    output_path='outputs/duke_dbt_froc_report.html',
    title='3D Mammography AI - FROC Localisation (Duke BCS-DBT, '
          + TEAM + ')',
    extra_limitations=[
        'Detections are a published DBTex-challenge submission, not a '
        'live device. The validation split has ~1,163 views and 75 '
        'biopsied-lesion boxes.',
    ])
HTML(open(path, encoding='utf-8').read())

## 7 · Notes & next steps
* **Validate your own detector.** Any DBT detector that emits a CSV with `StudyUID, View, X, Y, Width, Height, Z, Score` plugs straight in — convert it to `case_id, score, x, y, slice` (box corner → centre) and pass it to `ds.assemble_froc_cases(...)`. The FROC engine and the Duke hit criterion are unit-tested.
* **Held-out test set.** Use `split='test'` with the matching `BCS-DBT-*-test-*` CSVs and the `phase2/test` team predictions.
* **Exam-level classification.** Scoring the DICOM volumes themselves (cancer vs not) needs the ~1.5 TB image set: download a subset with `tcia_utils`, read volumes with the official `dcmread_image`, and wrap a 2D classifier in `SliceAggregatorClassifier`. That is a heavier, separate effort — the localisation validation above is a complete study on its own.